In [2]:
import random
import pandas as pd

In [12]:
def simulate_peachs_birthday_cake(turns=20, seed=None):
    """
    Simulates a 1-player 20-turn game on Peach's Birthday Cake (Mario Party 1).
    """
    if seed is not None:
        random.seed(seed)
        
    # Define Dice Spaces on the board layout
    DICE_SPACES = {1, 14, 21, 25, 35, 55}
    
    records = []
    current_space = 0
    total_roll_value = 0
    skip_next_turn = False
    global_roll_number = 0
    
    def step_forward(space):
        """Moves player forward 1 space, handling board forks and loops."""
        if space == 17:
            # 1 in 4 (25%) towards Bowser's Island; 3 in 4 (75%) towards Star/Toad path
            return 46 if random.random() < 0.25 else 18
        elif space == 45:
            # Reaches the end of either branch -> loops back to Start (Space 0)
            return 1
        elif space == 58:
            return 18
        else:
            return space + 1

    def advance_player(start_pos, steps):
        pos = start_pos
        for _ in range(steps):
            pos = step_forward(pos)
        return pos

    turn = 1
    while turn <= turns:
        turn_start_space = current_space
        
        # Handle skipped turn penalty
        if skip_next_turn:
            skip_next_turn = False
            global_roll_number += 1
            records.append({
                "Turn number": turn,
                "Roll Number": global_roll_number,
                "Roll Value": 0,
                "Is Extra Roll": False,
                "Is Lost Turn": True,
                "Space number at the start of the turn": turn_start_space,
                "Space number at the end of the turn": current_space,
                "total roll value so far": total_roll_value
            })
            turn += 1
            continue
            
        extra_rolls = 0
        roll_in_turn = 0
        
        while True:
            global_roll_number += 1
            is_extra = (roll_in_turn > 0)
            
            # Roll standard 1-10 die
            roll_val = random.randint(1, 10)
            total_roll_value += roll_val
            
            # Track space before individual roll
            space_at_roll_start = current_space
            current_space = advance_player(current_space, roll_val)
            
            records.append({
                "Turn number": turn,
                "Roll Number": global_roll_number,
                "Roll Value": roll_val,
                "Is Extra Roll": is_extra,
                "Is Lost Turn": False,
                "Space number at the start of the turn": turn_start_space,
                "Space number at the end of the turn": current_space,
                "total roll value so far": total_roll_value
            })
            
            # Check landing space effect (Dice Space / Mushroom)
            if current_space in DICE_SPACES:
                if random.random() < 0.5:
                    extra_rolls += 1  # Extra roll on this turn
                else:
                    skip_next_turn = True  # Lose next turn
            
            roll_in_turn += 1
            if extra_rolls > 0:
                extra_rolls -= 1
            else:
                break
                
        turn += 1

    return pd.DataFrame(records)

In [13]:
# Run 20-turn simulation
df_simulation = simulate_peachs_birthday_cake(turns=20)

# Set display options to show all rows cleanly
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Display DataFrame
df_simulation

,Turn number,Roll Number,Roll Value,Is Extra Roll,Is Lost Turn,Space number at the start of the turn,Space number at the end of the turn,total roll value so far
0,1,1,10,False,False,0,10,10
1,2,2,7,False,False,10,17,17
2,3,3,10,False,False,17,27,27
3,4,4,4,False,False,27,31,31
4,5,5,3,False,False,31,34,34
5,6,6,9,False,False,34,43,43
6,7,7,8,False,False,43,6,51
7,8,8,2,False,False,6,8,53
8,9,9,1,False,False,8,9,54
9,10,10,2,False,False,9,11,56
